# 09 · Multilayer Network on XOR

**Learning objectives**
1. Write the batch forward and backward pass in matrix form.
2. Check the matrix form against the per-sample loop and against finite differences.
3. Train a hidden layer on XOR and report how often training succeeds across seeds.

Modules: `src/mlp.py`, `tests/test_mlp.py`, `tests/test_multilayer_batch.py`. The 20-seed experiment is `experiments/multilayer_network/run.py`.

## 1. Why a hidden layer

Notebook 03 showed that one perceptron cannot represent XOR. A network with one hidden layer of sigmoid units can, and gradient descent can find weights that solve it. The output is trained with binary cross-entropy:

$$\1{L}=\1{1}{N}\1_{n}\big[-y_n\1\hat y_n-(1-y_n)\1(1-\hat y_n)\big]$$

## 2. Matrix form of the backward pass

With $X\1\1{R}^{N\1 2}$, $H=\1(XW_1+\mathbf{b}_1)$, $\hat Y=\1(HW_2+b_2)$:

$$dZ_2=\1{\hat Y-\mathbf{y}}{N},\quad dW_2=H^\1 dZ_2,\quad db_2=\1 dZ_2,$$
$$dH=dZ_2W_2^\1,\quad dZ_1=dH\1 H\1(1-H),\quad dW_1=X^\1 dZ_1,\quad db_1=\1 dZ_1$$

The first line uses the BCE-with-sigmoid cancellation from notebook 06. The factor $1/N$ comes from the mean loss.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

from mlp import init_network

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
Y = np.array([0, 1, 1, 0], dtype=float)
net = init_network(n_in=2, n_hidden=3, seed=0)

g_matrix, loss_matrix = net.batch_gradients(X, Y)
g_loop, loss_loop = net.batch_gradients_loop(X, Y)
print("loss (matrix) =", round(loss_matrix, 6), "  loss (loop) =", round(loss_loop, 6))
print("max difference between matrix and loop gradients:",
      max(float(np.abs(g_matrix[k] - g_loop[k]).max()) for k in g_matrix))

loss (matrix) = 0.698926   loss (loop) = 0.698926
max difference between matrix and loop gradients: 0.0


## 3. Finite-difference check on the batch loss

The same check as notebook 08, now on the batch loss.

In [2]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

eps = 1e-6
worst = 0.0
for name, arr in net.params().items():
    it = np.nditer(arr, flags=["multi_index"])
    for _ in it:
        idx = it.multi_index
        orig = arr[idx]
        arr[idx] = orig + eps; lp = net.batch_loss(X, Y)
        arr[idx] = orig - eps; lm = net.batch_loss(X, Y)
        arr[idx] = orig
        worst = max(worst, abs((lp - lm) / (2 * eps) - g_matrix[name][idx]))
print("max |finite difference - analytic| over all parameters:", f"{worst:.1e}")

max |finite difference - analytic| over all parameters: 5.6e-11


## 4. Train on XOR

Full-batch gradient descent, learning rate $0.5$, 3000 steps, three hidden units, seed 0.

In [3]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

net = init_network(n_in=2, n_hidden=3, seed=0)
history = net.fit(X, Y, lr=0.5, steps=3000)
print("loss: start", round(history[0], 4), " end", round(history[-1], 4))
print("predictions:", net.predict(X), " targets:", Y.astype(int))
print("accuracy:", net.accuracy(X, Y))

g = np.linspace(-0.5, 1.5, 200)
G1, G2 = np.meshgrid(g, g)
P = net.predict(np.c_[G1.ravel(), G2.ravel()]).reshape(G1.shape)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].contourf(G1, G2, P, levels=[-0.5, 0.5, 1.5], colors=["#f4cccc", "#cfe2f3"])
axes[0].scatter(X[:, 0], X[:, 1], c=Y, cmap="coolwarm", s=150, edgecolors="black", zorder=3)
axes[0].set_title("Learned decision regions (seed 0, 3 hidden units)")
axes[0].set_xlabel("x1"); axes[0].set_ylabel("x2")
axes[1].semilogy(history)
axes[1].set_title("Training loss"); axes[1].set_xlabel("step"); axes[1].grid(True, which="both")
plt.tight_layout(); plt.show()

loss: start 0.6989  end 0.0133
predictions: [0 1 1 0]  targets: [0 1 1 0]
accuracy: 1.0


## 5. How often does it succeed? (saved experiment)

`experiments/multilayer_network/run.py` repeats this for seeds 0 to 19 and widths 2, 3 and 4. The results are in `results/multilayer_xor.json`.

In [4]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

data = json.loads((ROOT / "results" / "multilayer_xor.json").read_text(encoding="utf-8"))
for s in data["summary"]:
    print(f"hidden={s['hidden_units']}: solved {s['solved']}/{s['runs']} seeds, median final loss {s['median_final_loss']:.4f}")

hidden=2: solved 16/20 seeds, median final loss 0.0135
hidden=3: solved 20/20 seeds, median final loss 0.0106
hidden=4: solved 20/20 seeds, median final loss 0.0087


## 6. Interpretation

- **Mathematical:** the matrix form computes the same gradients as the per-sample loop, and both match finite differences.
- **Experimental observation:** with two hidden units, some random starts end in a poor solution (16 of 20 succeeded). With three or four units, all 20 seeds succeeded in these runs.
- **Limitation:** this is one dataset, one learning rate and one step count. The success rate is not a general property of XOR networks.

## 7. Exercises

1. Set `n_hidden=1` and train with seed 0. Does it solve XOR? Explain with the decision regions.
2. Replace `1/N` by `1` in `dZ2`. What happens to the effective learning rate?
3. Run seeds 0 to 9 for `n_hidden=2`. Which seeds fail? Look at the final weights of a failed run.